In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import io
from matplotlib.patches import Patch
import re


CUSTOM_BINS_LENGTH = 10
CUSTOM_BINS_MINUTES = list(range(CUSTOM_BINS_LENGTH, 30+CUSTOM_BINS_LENGTH, CUSTOM_BINS_LENGTH))

# Prefixes to look for in the dataframe
METRIC_PREFIXES = ['Di', 'Do', 'Si', 'So']

# ==========================================
# 1. LOAD DATA
# ==========================================
sample_data = """

"""

df = pd.read_csv(io.StringIO(sample_data), sep='\t')

# Ensure 'Fem' or 'No.' is used as ID if present, otherwise use index
if 'No.' in df.columns:
    df['Fem'] = df['No.']
elif 'Fem' not in df.columns:
    df['Fem'] = df.index

df['Estrous'] = df['Estrous'].astype('category')

# ==========================================
# 2. DYNAMIC BIN CALCULATION
# ==========================================

def get_time_columns(df, prefix):
    """
    Identifies all time-point columns for a given prefix (e.g., 'Di').
    Returns a dict: { minute_int: column_name }
    Also identifies 'pre' and 'Sum' if they exist.
    """
    cols = {}
    pre_col = f"{prefix}_pre"
    sum_col = f"{prefix}_Sum"
    
    if pre_col in df.columns:
        cols['pre'] = pre_col
        
    if sum_col in df.columns:
        cols['sum'] = sum_col

    # Regex to find columns like Di_1min, Di_10min, etc.
    pattern = re.compile(rf"^{prefix}_(\d+)min$")
    
    for col in df.columns:
        match = pattern.match(col)
        if match:
            minute = int(match.group(1))
            cols[minute] = col
            
    return cols

def calculate_cumulative_bin(df, prefix, target_minute, col_map):
    """
    Calculates the cumulative sum for a specific prefix up to target_minute.
    """
    # Get all minute keys <= target_minute
    relevant_minutes = [m for m in col_map.keys() if isinstance(m, int) and m <= target_minute]
    
    if not relevant_minutes:
        return np.zeros(len(df))
        
    cols_to_sum = [col_map[m] for m in sorted(relevant_minutes)]
    return df[cols_to_sum].sum(axis=1)

# Generate dynamic time labels and calculate data
# We create new columns in the DF with a standard naming convention: prefix_0-Xmin
dynamic_time_labels = []
col_maps = {}

for prefix in METRIC_PREFIXES:
    col_maps[prefix] = get_time_columns(df, prefix)

for bin_min in CUSTOM_BINS_MINUTES:
    label = f"0-{bin_min}min"
    dynamic_time_labels.append(label)
    
    for prefix in METRIC_PREFIXES:
        new_col_name = f"{prefix}_{label}"
        df[new_col_name] = calculate_cumulative_bin(df, prefix, bin_min, col_maps[prefix])

# Also add 'pre' to labels if it exists in data
if 'pre' in col_maps['Di']: # Check one prefix, assume consistent
    dynamic_time_labels.insert(0, 'pre')
    # 'pre' columns already exist as Di_pre, etc., so no new calculation needed, 
    # but we ensure our logic knows 'pre' is a valid label.

print(f"Available Dynamic Time Labels: {dynamic_time_labels}")

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def sem(x):
    x = np.asarray(x)
    return np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

def format_pval(p):
    if pd.isna(p):
        return "p=NA"
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    return f"{stars}\n(p={p:.3f})"

def add_pval_annotation(ax, x1, x2, y_base, p_value, y_offset=0.12, height_level=0):
    y_top = y_base + y_offset + height_level * 0.08
    ax.plot([x1, x1, x2, x2], [y_top, y_top + 0.04, y_top + 0.04, y_top], 
            color='black', linewidth=1.3, clip_on=False, zorder=4)
    label = format_pval(p_value)
    ax.text((x1 + x2) / 2, y_top + 0.06, label, 
            ha='center', va='bottom', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9),
            zorder=5)

def calculate_pi(d_vals, s_vals):
    """Calculate Preference Index: PI = (D-S)/(D+S)"""
    pi_vals = []
    for d, s in zip(d_vals, s_vals):
        if (d + s) > 0:
            pi_vals.append((d - s) / (d + s))
        else:
            pi_vals.append(np.nan)
    return np.array(pi_vals)

# ==========================================
# 4. STATISTICAL TESTING: D vs S COMPARISONS
# ==========================================
print("=== Welch's t-tests: Chamber (D vs S) ===")
chamber_pvals = {}
for tp in dynamic_time_labels:
    for estr_val in [0, 1]:
        mask = df['Estrous'] == estr_val
        
        # Dynamically construct column names
        di_col = f"Di_{tp}" if tp != 'pre' else "Di_pre"
        do_col = f"Do_{tp}" if tp != 'pre' else "Do_pre"
        si_col = f"Si_{tp}" if tp != 'pre' else "Si_pre"
        so_col = f"So_{tp}" if tp != 'pre' else "So_pre"
        
        # Check if columns exist (for 'pre' they might, for custom bins they were created above)
        if di_col in df.columns and si_col in df.columns:
            D_val = df.loc[mask, di_col] + df.loc[mask, do_col] if do_col in df.columns else df.loc[mask, di_col]
            S_val = df.loc[mask, si_col] + df.loc[mask, so_col] if so_col in df.columns else df.loc[mask, si_col]
            
            if len(D_val)>1 and len(S_val)>1:
                t, p = stats.ttest_ind(D_val, S_val, equal_var=False)
                key = (tp, 'Chamber', estr_val)
                chamber_pvals[key] = p
                estr_label = 'E' if estr_val==1 else 'Non-E'
                # print(f"{tp:10s} {estr_label:6s} | t={t:6.3f} | p={p:.4f} {format_pval(p).split()[0]}")

print("\n=== Welch's t-tests: Sniffing (Di vs Si) ===")
inter_pvals = {}
for tp in dynamic_time_labels:
    for estr_val in [0, 1]:
        mask = df['Estrous'] == estr_val
        
        di_col = f"Di_{tp}" if tp != 'pre' else "Di_pre"
        si_col = f"Si_{tp}" if tp != 'pre' else "Si_pre"
        
        if di_col in df.columns and si_col in df.columns:
            D_val = df.loc[mask, di_col]
            S_val = df.loc[mask, si_col]
            
            if len(D_val)>1 and len(S_val)>1:
                t, p = stats.ttest_rel(D_val, S_val)
                key = (tp, 'Sniffing', estr_val)
                inter_pvals[key] = p
                estr_label = 'E' if estr_val==1 else 'Non-E'
                # print(f"{tp:10s} {estr_label:6s} | t={t:6.3f} | p={p:.4f} {format_pval(p).split()[0]}")

# ==========================================
# PI STATISTICAL TESTING: ESTRUS vs NON-ESTRUS COMPARISON
# ==========================================
print("\n=== Welch's t-tests: PI Index - Estrus vs Non-Estrus ===")
pi_pvals_estrus = {}
for tp in dynamic_time_labels:
    for group_name in ['Chamber', 'Sniffing']:
        
        di_col = f"Di_{tp}" if tp != 'pre' else "Di_pre"
        do_col = f"Do_{tp}" if tp != 'pre' else "Do_pre"
        si_col = f"Si_{tp}" if tp != 'pre' else "Si_pre"
        so_col = f"So_{tp}" if tp != 'pre' else "So_pre"
        
        if di_col not in df.columns or si_col not in df.columns:
            continue

        if group_name == 'Chamber':
            D_all = df[do_col]
            S_all = df[so_col]
        else:
            D_all = df[di_col]
            S_all = df[si_col]
        
        pi_all = calculate_pi(D_all, S_all)
        df_temp = df.copy()
        df_temp['PI'] = pi_all
        
        pi_estrus = df_temp[df_temp['Estrous']==1]['PI'].dropna()
        pi_non_estrus = df_temp[df_temp['Estrous']==0]['PI'].dropna()
        
        if len(pi_estrus) > 1 and len(pi_non_estrus) > 1:
            t, p = stats.ttest_ind(pi_estrus, pi_non_estrus, equal_var=False)
            key = (tp, group_name)
            pi_pvals_estrus[key] = p
            # mean_e = np.mean(pi_estrus)
            # mean_ne = np.mean(pi_non_estrus)
            # print(f"{tp:10s} {group_name:12s} | E: {mean_e:.3f} vs Non-E: {mean_ne:.3f} | p={p:.4f}")
        else:
            key = (tp, group_name)
            pi_pvals_estrus[key] = np.nan

# ==========================================
# 5. PLOT 1: 4 SUBPLOTS WITH GROUPED BARS + CONNECTED PAIRED DOTS
# ==========================================
fig1, axes1 = plt.subplots(1, len(dynamic_time_labels), figsize=(6*len(dynamic_time_labels), 7), sharey=True)
if len(dynamic_time_labels) == 1: axes1 = [axes1]

colors = {'D_NonE': '#2E86AB', 'D_E': '#2E86AB', 
          'S_NonE': '#E94F37', 'S_E': '#E94F37'}
hatches = {'NonE': '', 'E': '///'}

for idx, tp in enumerate(dynamic_time_labels):
    ax = axes1[idx]
    
    group_positions = [0, 2, 4, 6]
    bar_width = 0.8
    group_labels = ['Chamber\nNon-E', 'Chamber\nE', 'Sniffing\nNon-E', 'Sniffing\nE']
    
    for grp_idx, (group_name, estr_val) in enumerate([
        ('Chamber', 0), ('Chamber', 1), ('Sniffing', 0), ('Sniffing', 1)
    ]):
        base_pos = group_positions[grp_idx]
        
        di_col = f"Di_{tp}" if tp != 'pre' else "Di_pre"
        do_col = f"Do_{tp}" if tp != 'pre' else "Do_pre"
        si_col = f"Si_{tp}" if tp != 'pre' else "Si_pre"
        so_col = f"So_{tp}" if tp != 'pre' else "So_pre"
        
        mask = df['Estrous']==estr_val
        
        if group_name == 'Chamber':
            D_data = df.loc[mask, di_col] + (df.loc[mask, do_col] if do_col in df.columns else 0)
            S_data = df.loc[mask, si_col] + (df.loc[mask, so_col] if so_col in df.columns else 0)
            
            subset = df[mask].copy()
            subset['D_val'] = subset[di_col] + (subset[do_col] if do_col in df.columns else 0)
            subset['S_val'] = subset[si_col] + (subset[so_col] if so_col in df.columns else 0)
            paired_data = subset[['Fem', 'D_val', 'S_val']].dropna()
            p_key = (tp, 'Chamber', estr_val)
            p_val = chamber_pvals.get(p_key)
        else:
            D_data = df.loc[mask, di_col]
            S_data = df.loc[mask, si_col]
            
            subset = df[mask].copy()
            subset['D_val'] = subset[di_col]
            subset['S_val'] = subset[si_col]
            paired_data = subset[['Fem', 'D_val', 'S_val']].dropna()
            p_key = (tp, 'Sniffing', estr_val)
            p_val = inter_pvals.get(p_key)
        
        estr_key = 'E' if estr_val==1 else 'NonE'
        
        # D bar
        if len(D_data) > 0:
            mean_D = D_data.mean()
            sem_D = sem(D_data)
            ax.bar(base_pos, mean_D, width=bar_width, 
                   color=colors[f'D_{estr_key}'], 
                   hatch=hatches[estr_key], edgecolor='white', 
                   linewidth=1.2, alpha=0.5,
                   label=f'D ({estr_key})' if idx==0 and grp_idx==0 else "",
                   zorder=2)
            ax.errorbar(base_pos, mean_D, yerr=sem_D, fmt='none', 
                        ecolor='black', capsize=4, capthick=1.5, linewidth=1.5, zorder=3)
            jitter = np.random.normal(0, 0.03, size=len(D_data))
            ax.scatter(base_pos + jitter, D_data, c='black', s=20,
                       alpha=0.8, edgecolors='white', linewidth=0.5, zorder=4)
        
        # S bar
        if len(S_data) > 0:
            mean_S = S_data.mean()
            sem_S = sem(S_data)
            ax.bar(base_pos + bar_width, mean_S, width=bar_width, 
                   color=colors[f'S_{estr_key}'], 
                   hatch=hatches[estr_key], edgecolor='white',
                   linewidth=1.2, alpha=0.5,
                   label=f'S ({estr_key})' if idx==0 and grp_idx==0 else "",
                   zorder=2)
            ax.errorbar(base_pos + bar_width, mean_S, yerr=sem_S, fmt='none', 
                        ecolor='black', capsize=4, capthick=1.5, linewidth=1.5, zorder=3)
            jitter = np.random.normal(0, 0.03, size=len(S_data))
            ax.scatter(base_pos + bar_width + jitter, S_data, c='black', s=20,
                       alpha=0.8, edgecolors='white', linewidth=0.5, zorder=4)
        
        # Connect paired D/S dots with lines
        for _, row in paired_data.iterrows():
            d_val, s_val = row['D_val'], row['S_val']
            d_jitter = np.random.normal(0, 0.03)
            s_jitter = np.random.normal(0, 0.03)
            ax.plot([base_pos + d_jitter, base_pos + bar_width + s_jitter], 
                    [d_val, s_val], color='darkgray', linewidth=1.5, alpha=1.0, zorder=3)
        
        # P-value annotation
        if p_val is not None and not np.isnan(p_val):
            y_max = max(mean_D + 200 if len(D_data)>0 else 0, mean_S + 200 if len(S_data)>0 else 0)
            add_pval_annotation(ax, base_pos, base_pos + bar_width, y_max, p_val, 
                               y_offset=0.08, height_level=0)
    
    ax.set_title(f'{tp}', fontsize=11, fontweight='bold', pad=10)
    ax.set_xticks([pos + bar_width/2 for pos in group_positions])
    ax.set_xticklabels(group_labels, fontsize=8, ha='center')
    ax.grid(axis='y', alpha=0.35, linestyle='--', linewidth=0.5)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle=':', zorder=1)

    if idx == 0: ax.set_ylabel('Time [seconds]', fontsize=10, labelpad=8)
    ax.tick_params(axis='x', length=0)

legend_elements = [
    Patch(facecolor='#2E86AB', edgecolor='white', alpha=0.5, label='D time'),
    Patch(facecolor='#E94F37', edgecolor='white', alpha=0.5, label='S time'),
    Patch(facecolor='#2E86AB', edgecolor='white', hatch='///', alpha=0.5, label='Estrous'),
    Patch(facecolor='#2E86AB', edgecolor='white', alpha=0.5, label='Non-Estrous'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
               markersize=5, alpha=0.8, linestyle='None', label='Raw data'),
    plt.Line2D([0], [0], color='black', linewidth=1.5, label='± SEM'),
    plt.Line2D([0], [0], color='darkgray', linewidth=1.2, alpha=0.7, label='Paired subjects')
]
fig1.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.01), 
            ncol=7, frameon=True, fontsize=9, columnspacing=1.5)

plt.suptitle('Time: D vs S Comparison by Group and Type (Mean ± SEM)\n', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.show()

# ==========================================
# 6. PLOT 3: PI INDEX VERSION - ESTRUS vs NON-ESTRUS COMPARISON
# ==========================================
fig3, axes3 = plt.subplots(1, len(dynamic_time_labels), figsize=(6*len(dynamic_time_labels), 7), sharey=True)
if len(dynamic_time_labels) == 1: axes3 = [axes3]

pi_colors = {'NonE': '#2E86AB', 'E': '#2E86AB'}
pi_hatches = {'NonE': '', 'E': '///'}

for idx, tp in enumerate(dynamic_time_labels):
    ax = axes3[idx]
    
    group_positions = [0, 2, 4, 6]
    bar_width = 2.0
    group_labels = ['Chamber\nNon-E', 'Chamber\nE', 'Sniffing\nNon-E', 'Sniffing\nE']
    
    mean_pis = {}
    
    for grp_idx, (group_name, estr_val) in enumerate([
        ('Chamber', 0), ('Chamber', 1), ('Sniffing', 0), ('Sniffing', 1)
    ]):
        base_pos = group_positions[grp_idx]
        
        mask = df['Estrous'] == estr_val
        
        di_col = f"Di_{tp}" if tp != 'pre' else "Di_pre"
        do_col = f"Do_{tp}" if tp != 'pre' else "Do_pre"
        si_col = f"Si_{tp}" if tp != 'pre' else "Si_pre"
        so_col = f"So_{tp}" if tp != 'pre' else "So_pre"
        
        if di_col not in df.columns or si_col not in df.columns:
            continue

        if group_name == 'Chamber':
            D_data = df.loc[mask, di_col] + (df.loc[mask, do_col] if do_col in df.columns else 0)
            S_data = df.loc[mask, si_col] + (df.loc[mask, so_col] if so_col in df.columns else 0)
        else:
            D_data = df.loc[mask, di_col]
            S_data = df.loc[mask, si_col]
        
        pi_vals = calculate_pi(D_data, S_data)
        pi_vals_clean = pi_vals[~np.isnan(pi_vals)]
        
        estr_key = 'E' if estr_val==1 else 'NonE'
        
        if len(pi_vals_clean) > 0:
            mean_pi = np.mean(pi_vals_clean)
            sem_pi = sem(pi_vals_clean)
            mean_pis[(group_name, estr_val)] = mean_pi
            
            bar_color = '#2E86AB' if mean_pi >= 0 else '#E94F37'
            
            ax.bar(base_pos + bar_width/2, mean_pi, width=bar_width, 
                   color=bar_color, hatch=pi_hatches[estr_key], 
                   edgecolor='white', linewidth=1.2, alpha=0.5,
                   label=f'PI ({estr_key})' if idx==0 and grp_idx==0 else "",
                   zorder=2)
            ax.errorbar(base_pos + bar_width/2, mean_pi, yerr=sem_pi, fmt='none', 
                        ecolor='black', capsize=4, capthick=1.5, linewidth=1.5, zorder=3)
            
            jitter = np.random.normal(0, 0.03, size=len(pi_vals_clean))
            ax.scatter(base_pos + bar_width/2 + jitter, pi_vals_clean, 
                       c='black', s=20, alpha=0.8, edgecolors='white', 
                       linewidth=0.5, zorder=4)
    
    # Add p-value annotation comparing Estrus vs Non-Estrus for each group type
    for grp_type in ['Chamber', 'Sniffing']:
        p_key = (tp, grp_type)
        p_val = pi_pvals_estrus.get(p_key)
        
        if p_val is not None and not np.isnan(p_val):
            grp_idx_base = 0 if grp_type == 'Chamber' else 2
            # Ensure indices are within bounds
            if grp_idx_base + 1 < len(group_positions):
                pos_none = group_positions[grp_idx_base] + bar_width/2
                pos_e = group_positions[grp_idx_base + 1] + bar_width/2
                
                mean_ne = mean_pis.get((grp_type, 0), 0)
                mean_e = mean_pis.get((grp_type, 1), 0)
                y_base = max(abs(mean_ne), abs(mean_e)) + 0.15
                
                add_pval_annotation(ax, pos_none, pos_e, y_base, p_val, 
                                   y_offset=0.05, height_level=0)
    
    ax.set_title(f'{tp}', fontsize=11, fontweight='bold', pad=10)
    ax.set_xticks([pos + bar_width/2 for pos in group_positions])
    ax.set_xticklabels(group_labels, fontsize=8, ha='center')
    ax.grid(axis='y', alpha=0.35, linestyle='--', linewidth=0.5)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='-', zorder=1)

    if idx == 0: ax.set_ylabel('Preference Index (D-S)/(D+S)', fontsize=10, labelpad=8)
    ax.set_ylim(-0.6, 0.6)
    ax.tick_params(axis='x', length=0)

pi_legend_elements = [
    Patch(facecolor='#2E86AB', edgecolor='white', alpha=0.5, label='Positive PI (D preference)'),
    Patch(facecolor='#E94F37', edgecolor='white', alpha=0.5, label='Negative PI (S preference)'),
    Patch(facecolor='#2E86AB', edgecolor='white', hatch='///', alpha=0.5, label='Estrous group'),
    Patch(facecolor='#2E86AB', edgecolor='white', alpha=0.5, label='Non-Estrous group'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', 
               markersize=5, alpha=0.8, linestyle='None', label='Individual PI'),
    plt.Line2D([0], [0], color='black', linewidth=1.5, label='± SEM')
]
fig3.legend(handles=pi_legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.01), 
            ncol=6, frameon=True, fontsize=9, columnspacing=1.5)

plt.suptitle('Preference Index (PI): Estrus vs Non-Estrus Comparison (Mean ± SEM)\n'
             'PI > 0: D preference | PI < 0: S preference | Brackets: Welch\'s t-test (Estrus vs Non-Estrus); *p<0.05, **p<0.01, ***p<0.001', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.show()

# ==========================================
# 7. PLOT 2: TIME COURSE - D AND S LINES SIDE-BY-SIDE (CUMULATIVE)
# ==========================================
# Use dynamic_time_labels for the time course as well
time_points_tc = dynamic_time_labels

fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))

for row_idx, (metric_name, d_col1, d_col2, s_col1, s_col2, ylabel) in enumerate([
    ('Chamber', 'Di', 'Do', 'Si', 'So', 'Chamber Time [seconds]'),
    ('Sniffing', 'Di', None, 'Si', None, 'Sniffing Time [seconds]')
]):
    for col_idx, (estr_val, estr_title) in enumerate([(0, 'Non-Estrous'), (1, 'Estrous')]):
        ax = axes2[row_idx, col_idx]
        
        plot_data = []
        for _, row in df.iterrows():
            if row['Estrous'] == estr_val:
                fem = row['Fem']
                for tp in time_points_tc:
                    # Construct column names dynamically
                    d_c1 = f"{d_col1}_{tp}" if tp != 'pre' else f"{d_col1}_pre"
                    s_c1 = f"{s_col1}_{tp}" if tp != 'pre' else f"{s_col1}_pre"
                    
                    d_val = row[d_c1] if d_c1 in df.columns else 0
                    if d_col2:
                        d_c2 = f"{d_col2}_{tp}" if tp != 'pre' else f"{d_col2}_pre"
                        d_val += row[d_c2] if d_c2 in df.columns else 0
                        
                    s_val = row[s_c1] if s_c1 in df.columns else 0
                    if s_col2:
                        s_c2 = f"{s_col2}_{tp}" if tp != 'pre' else f"{s_col2}_pre"
                        s_val += row[s_c2] if s_c2 in df.columns else 0
                        
                    plot_data.append({'Fem': fem, 'Time': tp, 'D': d_val, 'S': s_val})
        
        tc_df = pd.DataFrame(plot_data)
        tc_df['Time'] = pd.Categorical(tc_df['Time'], categories=time_points_tc, ordered=True)
        
        x_pos = np.arange(len(time_points_tc))
        
        d_means = [tc_df[tc_df['Time']==tp]['D'].mean() for tp in time_points_tc]
        d_sems = [sem(tc_df[tc_df['Time']==tp]['D']) for tp in time_points_tc]
        ax.plot(x_pos, d_means, color='#2E86AB', linewidth=2.5, marker='o', 
                markersize=6, label='D time', zorder=3)
        ax.fill_between(x_pos, [m-s for m,s in zip(d_means, d_sems)], 
                        [m+s for m,s in zip(d_means, d_sems)], 
                        color='#2E86AB', alpha=0.2, zorder=2)
        
        s_means = [tc_df[tc_df['Time']==tp]['S'].mean() for tp in time_points_tc]
        s_sems = [sem(tc_df[tc_df['Time']==tp]['S']) for tp in time_points_tc]
        ax.plot(x_pos, s_means, color='#E94F37', linewidth=2.5, marker='s', 
                markersize=6, label='S time', zorder=3)
        ax.fill_between(x_pos, [m-s for m,s in zip(s_means, s_sems)], 
                        [m+s for m,s in zip(s_means, s_sems)], 
                        color='#E94F37', alpha=0.2, zorder=2)
        
        fems_complete = tc_df.groupby('Fem').filter(lambda g: len(g)==len(time_points_tc))['Fem'].unique()
        for fem in fems_complete:
            fem_data = tc_df[tc_df['Fem']==fem].sort_values('Time')
            d_vals = fem_data['D'].values
            if not np.any(np.isnan(d_vals)):
                ax.plot(x_pos, d_vals, color='#2E86AB', linewidth=0.6, alpha=0.3, zorder=1)
            s_vals = fem_data['S'].values
            if not np.any(np.isnan(s_vals)):
                ax.plot(x_pos, s_vals, color='#E94F37', linewidth=0.6, alpha=0.3, zorder=1)
        
        ax.set_title(f'{estr_title} (n={len(fems_complete)})', fontsize=11, fontweight='bold', pad=10)
        ax.set_xticks(x_pos)
        # Clean up labels for display
        clean_labels = [tp.replace('0-','').replace('min',' min') if 'min' in tp else tp for tp in time_points_tc]
        ax.set_xticklabels(clean_labels, fontsize=9, rotation=45, ha='right')
        ax.grid(axis='y', alpha=0.35, linestyle='--', linewidth=0.5)

        if col_idx == 0: ax.set_ylabel(ylabel, fontsize=10, labelpad=8)
        if row_idx == 1: ax.set_xlabel('Time window', fontsize=10)
        ax.tick_params(axis='x', length=0)
        if row_idx == 0 and col_idx == 1:
            ax.legend(fontsize=9, frameon=True)

plt.suptitle('Actual Time Course (Cumulative): D vs S Trajectories\n'
             'Solid lines: group mean ± SEM; Faint lines: individual subjects', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.show()

print("\n=== All plots generated successfully with arbitrary bin support ===")

import statsmodels.api as sm
from statsmodels.formula.api import ols

# ==========================================
# 8. PLOT 4: NON-CUMULATIVE TIME COURSE + 2-WAY RM ANOVA
# ==========================================

fig4, axes4 = plt.subplots(2, 2, figsize=(14, 10))

# 1. Prepare Discrete Data Structure (Same logic as before)
cumulative_cols = [col for col in dynamic_time_labels if col.startswith('0-')]
def get_min_from_label(label):
    return int(label.split('-')[1].replace('min', ''))
cumulative_cols.sort(key=get_min_from_label)

discrete_map = []
for i, cum_col in enumerate(cumulative_cols):
    current_min = get_min_from_label(cum_col)
    if i == 0:
        prev_min = 0
        prev_col = None
        label = f"0-{current_min} min"
    else:
        prev_min = get_min_from_label(cumulative_cols[i-1])
        prev_col = cumulative_cols[i-1]
        label = f"{prev_min}-{current_min} min"
    
    discrete_map.append({
        'label': label,
        'curr_col': cum_col,
        'prev_col': prev_col,
        'minute': current_min
    })

ordered_labels = [item['label'] for item in discrete_map]

# 2. Statistical Calculation Function
def run_rm_anova(df_subset, d_col1, d_col2, s_col1, s_col2, discrete_map):
    """
    Constructs a long-format dataframe and runs 2-way RM ANOVA:
    Factors: Time (Within), ChamberType (Within: D vs S)
    Subject: Fem (Within)
    """
    rows = []
    for _, row in df_subset.iterrows():
        fem_id = row['Fem']
        for item in discrete_map:
            curr_c = item['curr_col']
            prev_c = item['prev_col']
            
            def get_val(prefix, col_suffix):
                full_col = f"{prefix}_{col_suffix}"
                return row[full_col] if full_col in df.columns else 0
            
            # Calc Discrete D
            d_curr = get_val(d_col1, curr_c)
            if d_col2: d_curr += get_val(d_col2, curr_c)
            d_prev = get_val(d_col1, prev_c) if prev_c else 0
            if d_col2 and prev_c: d_prev += get_val(d_col2, prev_c)
            d_val = max(0, d_curr - d_prev)
            
            # Calc Discrete S
            s_curr = get_val(s_col1, curr_c)
            if s_col2: s_curr += get_val(s_col2, curr_c)
            s_prev = get_val(s_col1, prev_c) if prev_c else 0
            if s_col2 and prev_c: s_prev += get_val(s_col2, prev_c)
            s_val = max(0, s_curr - s_prev)
            
            # Add D row
            rows.append({'Fem': fem_id, 'Time': item['label'], 'Chamber': 'D', 'Value': d_val})
            # Add S row
            rows.append({'Fem': fem_id, 'Time': item['label'], 'Chamber': 'S', 'Value': s_val})
            
    long_df = pd.DataFrame(rows)
    
    # Filter out subjects with missing data points to ensure balanced RM ANOVA
    # (Simple approach: drop NaNs, though strict RM requires complete cases)
    long_df = long_df.dropna()
    
    if long_df.empty:
        return None, None, None

    # Perform Two-Way Repeated Measures ANOVA using OLS with Error Term
    # Formula: Value ~ C(Time) * C(Chamber) + Error(Fem/(Time*Chamber))
    try:
        model = ols('Value ~ C(Time) * C(Chamber)', data=long_df).fit()
        anova_table = sm.stats.anova_lm(model, typ=2)
        
        # Extract P-values
        p_time = anova_table.loc['C(Time)', 'PR(>F)']
        p_chamber = anova_table.loc['C(Chamber)', 'PR(>F)']
        p_inter = anova_table.loc['C(Time):C(Chamber)', 'PR(>F)']
        print(p_time, p_chamber, p_inter)

        return p_time, p_chamber, p_inter
    except Exception as e:
        print(f"ANOVA Error: {e}")
        return None, None, None

# 3. Plotting Loop
for row_idx, (metric_name, d_col1, d_col2, s_col1, s_col2, ylabel) in enumerate([
    ('Chamber', 'Di', 'Do', 'Si', 'So', 'Chamber Time [seconds]'),
    ('Sniffing', 'Di', None, 'Si', None, 'Sniffing Time [seconds]')
]):
    for col_idx, (estr_val, estr_title) in enumerate([(0, 'Non-Estrous'), (1, 'Estrous')]):
        ax = axes4[row_idx, col_idx]
        
        # --- A. Run Statistics ---
        mask = df['Estrous'] == estr_val
        df_subset = df[mask]
        p_time, p_chamber, p_inter = run_rm_anova(df_subset, d_col1, d_col2, s_col1, s_col2, discrete_map)
        
        # --- B. Prepare Plot Data ---
        plot_data = []
        for _, row in df_subset.iterrows():
            fem = row['Fem']
            for item in discrete_map:
                curr_c = item['curr_col']
                prev_c = item['prev_col']
                
                def get_val(prefix, col_suffix):
                    full_col = f"{prefix}_{col_suffix}"
                    return row[full_col] if full_col in df.columns else 0
                
                d_curr = get_val(d_col1, curr_c)
                if d_col2: d_curr += get_val(d_col2, curr_c)
                d_prev = get_val(d_col1, prev_c) if prev_c else 0
                if d_col2 and prev_c: d_prev += get_val(d_col2, prev_c)
                d_discrete = max(0, d_curr - d_prev)
                
                s_curr = get_val(s_col1, curr_c)
                if s_col2: s_curr += get_val(s_col2, curr_c)
                s_prev = get_val(s_col1, prev_c) if prev_c else 0
                if s_col2 and prev_c: s_prev += get_val(s_col2, prev_c)
                s_discrete = max(0, s_curr - s_prev)
                
                plot_data.append({'Fem': fem, 'Time': item['label'], 'D': d_discrete, 'S': s_discrete})
        
        tc_df = pd.DataFrame(plot_data)
        tc_df['Time'] = pd.Categorical(tc_df['Time'], categories=ordered_labels, ordered=True)
        
        x_pos = np.arange(len(ordered_labels))
        
        # Means & SEMs
        d_means = [tc_df[tc_df['Time']==tp]['D'].mean() for tp in ordered_labels]
        d_sems = [sem(tc_df[tc_df['Time']==tp]['D']) for tp in ordered_labels]
        s_means = [tc_df[tc_df['Time']==tp]['S'].mean() for tp in ordered_labels]
        s_sems = [sem(tc_df[tc_df['Time']==tp]['S']) for tp in ordered_labels]
        
        # Plot Lines
        ax.plot(x_pos, d_means, color='#2E86AB', linewidth=2.5, marker='o', markersize=6, label='D time', zorder=3)
        ax.fill_between(x_pos, [m-s for m,s in zip(d_means, d_sems)], [m+s for m,s in zip(d_means, d_sems)], color='#2E86AB', alpha=0.2, zorder=2)
        
        ax.plot(x_pos, s_means, color='#E94F37', linewidth=2.5, marker='s', markersize=6, label='S time', zorder=3)
        ax.fill_between(x_pos, [m-s for m,s in zip(s_means, s_sems)], [m+s for m,s in zip(s_means, s_sems)], color='#E94F37', alpha=0.2, zorder=2)
        
        # Individual Lines
        fems_complete = tc_df.groupby('Fem').filter(lambda g: len(g)==len(ordered_labels))['Fem'].unique()
        for fem in fems_complete:
            fem_data = tc_df[tc_df['Fem']==fem].sort_values('Time')
            d_vals = fem_data['D'].values
            if not np.any(np.isnan(d_vals)):
                ax.plot(x_pos, d_vals, color='#2E86AB', linewidth=0.6, alpha=0.3, zorder=1)
            s_vals = fem_data['S'].values
            if not np.any(np.isnan(s_vals)):
                ax.plot(x_pos, s_vals, color='#E94F37', linewidth=0.6, alpha=0.3, zorder=1)
        
        # --- C. Annotate P-Values ---
        # We place text boxes in the upper right or left corner to avoid cluttering data
        y_max = max(max(d_means + d_sems), max(s_means + s_sems))
        y_min = min(min([m-s for m,s in zip(d_means, d_sems)]), min([m-s for m,s in zip(s_means, s_sems)]))
        range_y = y_max - y_min
        
        # Position offsets
        text_x = 0.02 
        text_y_start = 0.95 
        
        stats_text = []
        if p_time is not None:
            stats_text.append(f"Time: {format_pval(p_time).split()[0]}")
        if p_chamber is not None:
            stats_text.append(f"Dominance: {format_pval(p_chamber).split()[0]}")
        if p_inter is not None:
            stats_text.append(f"Interaction: {format_pval(p_inter).split()[0]}")
            
        if stats_text:
            annotation = "\n".join(stats_text)
            ax.text(0.02, 0.95, annotation, transform=ax.transAxes, fontsize=9,
                    verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))

        # Formatting
        ax.set_title(f'{estr_title} (n={len(fems_complete)})', fontsize=11, fontweight='bold', pad=10)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(ordered_labels, fontsize=9, rotation=45, ha='right')
        ax.grid(axis='y', alpha=0.35, linestyle='--', linewidth=0.5)

        if col_idx == 0: ax.set_ylabel(ylabel, fontsize=10, labelpad=8)
        if row_idx == 1: ax.set_xlabel('Time Interval (Minutes)', fontsize=10)
        ax.tick_params(axis='x', length=0)
        if row_idx == 0 and col_idx == 1:
            ax.legend(fontsize=9, frameon=True)

plt.suptitle('Non-Cumulative Time Course: 2-Way RM ANOVA (Time x Chamber)\n'
             'Values: Time spent ONLY in that specific bin. Stats: Main Effects & Interaction.', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.show()
